# MOVEとDirectionへの分解・選択的売買

**Historical archive / 過去の研究記録**

原本のコードを保持しています。独立実行や現在の検証基準への適合は保証しません。前のセルの変数に依存する箇所があります。実行入口は `../08_trade_quality.ipynb` を参照してください。

保存出力は `../../results/legacy/`、既知の問題は `../../docs/AUDIT.md` に整理しています。


## 元Notebookのセル 6

出典: `FX.ipynb`、0始まりのindex=5。コード内容は変更していません。

In [ ]:
# =========================================
# 1. ライブラリ読み込み
# =========================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report


# =========================================
# 2. USD/JPY 5分足データ取得
# =========================================

df = yf.download(
    "JPY=X",
    period="60d",
    interval="5m",
    auto_adjust=False
)

# yfinanceの列名が2段になっていたら1段に直す
if df.columns.nlevels > 1:
    df.columns = df.columns.get_level_values(0)

print("取得データ数:", len(df))


# =========================================
# 3. 基本的な値動きの特徴量
# =========================================

# 5分リターン
df["return_5m"] = df["Close"].pct_change(1)

# 15分リターン
df["return_15m"] = df["Close"].pct_change(3)

# 30分リターン
df["return_30m"] = df["Close"].pct_change(6)

# 1時間リターン
df["return_1h"] = df["Close"].pct_change(12)


# =========================================
# 4. 移動平均
# =========================================

df["MA5"] = df["Close"].rolling(5).mean()
df["MA20"] = df["Close"].rolling(20).mean()
df["MA50"] = df["Close"].rolling(50).mean()

# 現在価格と移動平均の距離
df["MA5_distance"] = df["Close"] / df["MA5"] - 1
df["MA20_distance"] = df["Close"] / df["MA20"] - 1
df["MA50_distance"] = df["Close"] / df["MA50"] - 1

# MA20の傾き
df["MA20_slope"] = df["MA20"].pct_change(3)


# =========================================
# 5. ローソク足の特徴
# =========================================

# ローソクの実体
df["body"] = abs(
    df["Close"] - df["Open"]
) / df["Open"]

# 全体の値幅
df["range"] = (
    df["High"] - df["Low"]
) / df["Close"]

# 上ヒゲ
df["upper_wick"] = (
    df["High"]
    - df[["Open", "Close"]].max(axis=1)
) / df["Close"]

# 下ヒゲ
df["lower_wick"] = (
    df[["Open", "Close"]].min(axis=1)
    - df["Low"]
) / df["Close"]

# 陽線なら1、陰線なら0
df["bullish"] = (
    df["Close"] > df["Open"]
).astype(int)


# =========================================
# 6. ボラティリティ
# =========================================

# 過去1時間の標準偏差
df["volatility_1h"] = (
    df["return_5m"]
    .rolling(12)
    .std()
)

# 過去2時間
df["volatility_2h"] = (
    df["return_5m"]
    .rolling(24)
    .std()
)


# =========================================
# 7. RSIを作る
# =========================================

delta = df["Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df["RSI"] = 100 - (
    100 / (1 + rs)
)


# =========================================
# 8. 直近高値・安値からの距離
# =========================================

df["high_1h"] = df["High"].rolling(12).max()
df["low_1h"] = df["Low"].rolling(12).min()

df["distance_from_high_1h"] = (
    df["Close"] / df["high_1h"] - 1
)

df["distance_from_low_1h"] = (
    df["Close"] / df["low_1h"] - 1
)


# =========================================
# 9. 時刻・曜日
# =========================================

df["hour"] = df.index.hour
df["weekday"] = df.index.dayofweek


# =========================================
# 10. 30分後リターン
# =========================================

df["future_return_30m"] = (
    df["Close"].shift(-6)
    / df["Close"]
    - 1
)


# =========================================
# 11. 大きく動いた時だけ正解データにする
# =========================================

threshold = 0.0005
# 0.0005 = 0.05%

df["target"] = np.where(
    df["future_return_30m"] > threshold,
    1,
    np.where(
        df["future_return_30m"] < -threshold,
        0,
        np.nan
    )
)


# =========================================
# 12. AIに渡す特徴量
# =========================================

features = [
    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",

    "MA5_distance",
    "MA20_distance",
    "MA50_distance",
    "MA20_slope",

    "body",
    "range",
    "upper_wick",
    "lower_wick",
    "bullish",

    "volatility_1h",
    "volatility_2h",

    "RSI",

    "distance_from_high_1h",
    "distance_from_low_1h",

    "hour",
    "weekday"
]


# =========================================
# 13. 欠損データを除外
# =========================================

data = df[
    features
    + ["target", "future_return_30m"]
].dropna()

# targetを整数型へ
data["target"] = data["target"].astype(int)

print("学習対象データ数:", len(data))


# =========================================
# 14. 時間順に学習・テスト分割
# =========================================

split = int(len(data) * 0.8)

train = data.iloc[:split]
test = data.iloc[split:]

X_train = train[features]
y_train = train["target"]

X_test = test[features]
y_test = test["target"]

print("学習データ数:", len(train))
print("テストデータ数:", len(test))


# =========================================
# 15. Random Forestモデル
# =========================================

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=15,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


# =========================================
# 16. 学習
# =========================================

model.fit(
    X_train,
    y_train
)


# =========================================
# 17. テストデータ予測
# =========================================

pred = model.predict(X_test)

prob = model.predict_proba(
    X_test
)[:, 1]


# =========================================
# 18. 性能評価
# =========================================

accuracy = accuracy_score(
    y_test,
    pred
)

auc = roc_auc_score(
    y_test,
    prob
)

print("\n========================")
print("モデル評価")
print("========================")

print(
    "Accuracy:",
    round(accuracy, 4)
)

print(
    "ROC-AUC:",
    round(auc, 4)
)

print("\n")
print(
    classification_report(
        y_test,
        pred
    )
)


# =========================================
# 19. 高確率シグナルだけ評価
# =========================================

results = test.copy()

results["up_probability"] = prob
results["prediction"] = pred

# 60%以上ならBUY候補
buy_signals = results[
    results["up_probability"] >= 0.60
]

# 40%以下ならSELL候補
sell_signals = results[
    results["up_probability"] <= 0.40
]

print("========================")
print("高確率シグナル")
print("========================")

print(
    "BUY候補数:",
    len(buy_signals)
)

print(
    "SELL候補数:",
    len(sell_signals)
)


# =========================================
# 20. BUYシグナルの実際の成績
# =========================================

if len(buy_signals) > 0:

    buy_win_rate = (
        buy_signals["target"] == 1
    ).mean()

    print(
        "BUYシグナル勝率:",
        round(
            buy_win_rate * 100,
            2
        ),
        "%"
    )


# =========================================
# 21. SELLシグナルの実際の成績
# =========================================

if len(sell_signals) > 0:

    sell_win_rate = (
        sell_signals["target"] == 0
    ).mean()

    print(
        "SELLシグナル勝率:",
        round(
            sell_win_rate * 100,
            2
        ),
        "%"
    )


# =========================================
# 22. 最新の上昇確率
# =========================================

latest_X = data[
    features
].iloc[[-1]]

latest_probability = (
    model.predict_proba(
        latest_X
    )[0, 1]
)

print("========================")
print("最新予測")
print("========================")

print(
    "30分後の上昇確率:",
    round(
        latest_probability * 100,
        2
    ),
    "%"
)


# =========================================
# 23. 予測確率グラフ
# =========================================

plt.figure(
    figsize=(14, 6)
)

plt.plot(
    results.index,
    results["up_probability"]
)

plt.axhline(
    0.60,
    linestyle="--"
)

plt.axhline(
    0.40,
    linestyle="--"
)

plt.title(
    "USD/JPY 30-minute Direction Probability"
)

plt.xlabel("Time")
plt.ylabel("Up Probability")

plt.grid()

plt.show()


# =========================================
# 24. 特徴量重要度
# =========================================

importance = pd.Series(
    model.feature_importances_,
    index=features
)

importance = importance.sort_values(
    ascending=False
)

print("========================")
print("特徴量重要度")
print("========================")

print(importance)

importance.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title(
    "Feature Importance"
)

plt.ylabel(
    "Importance"
)

plt.grid()

plt.show()

## 元Notebookのセル 7

出典: `FX.ipynb`、0始まりのindex=6。コード内容は変更していません。

In [ ]:
# =========================================
# 25. 予測確率帯ごとの実勝率を確認
# =========================================

# 上昇確率を5%刻みで分類する
bins = [
    0.00,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    1.00
]

labels = [
    "0-40%",
    "40-45%",
    "45-50%",
    "50-55%",
    "55-60%",
    "60-65%",
    "65-70%",
    "70-75%",
    "75-100%"
]

results["probability_group"] = pd.cut(
    results["up_probability"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# 各確率帯について
# 件数と実際の上昇率を計算
probability_analysis = (
    results
    .groupby(
        "probability_group",
        observed=False
    )
    .agg(
        count=("target", "size"),
        actual_up_rate=("target", "mean"),
        average_return=("future_return_30m", "mean")
    )
)

# パーセント表示にする
probability_analysis["actual_up_rate"] *= 100
probability_analysis["average_return"] *= 100

print("\n========================")
print("予測確率帯ごとの実績")
print("========================")

print(probability_analysis)


# =========================================
# 26. BUYシグナルのリターン分析
# =========================================

buy_results = results[
    results["up_probability"] >= 0.60
].copy()

print("\n========================")
print("BUYシグナル分析")
print("========================")

if len(buy_results) > 0:

    print(
        "件数:",
        len(buy_results)
    )

    print(
        "勝率:",
        round(
            (
                buy_results["future_return_30m"] > 0
            ).mean() * 100,
            2
        ),
        "%"
    )

    print(
        "平均30分リターン:",
        round(
            buy_results[
                "future_return_30m"
            ].mean() * 100,
            4
        ),
        "%"
    )

    print(
        "中央値30分リターン:",
        round(
            buy_results[
                "future_return_30m"
            ].median() * 100,
            4
        ),
        "%"
    )

    print(
        "最大利益:",
        round(
            buy_results[
                "future_return_30m"
            ].max() * 100,
            4
        ),
        "%"
    )

    print(
        "最大損失:",
        round(
            buy_results[
                "future_return_30m"
            ].min() * 100,
            4
        ),
        "%"
    )


# =========================================
# 27. 閾値ごとの成績比較
# =========================================

thresholds = [
    0.55,
    0.60,
    0.65,
    0.70,
    0.75
]

threshold_results = []

for threshold_prob in thresholds:

    trades = results[
        results["up_probability"]
        >= threshold_prob
    ]

    if len(trades) == 0:
        continue

    win_rate = (
        trades["future_return_30m"] > 0
    ).mean()

    average_return = (
        trades["future_return_30m"]
        .mean()
    )

    median_return = (
        trades["future_return_30m"]
        .median()
    )

    threshold_results.append({
        "threshold": threshold_prob,
        "trades": len(trades),
        "win_rate": win_rate,
        "average_return": average_return,
        "median_return": median_return
    })


threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df["win_rate"] *= 100
threshold_df["average_return"] *= 100
threshold_df["median_return"] *= 100

print("\n========================")
print("BUY確率閾値比較")
print("========================")

print(threshold_df)


# =========================================
# 28. スプレッドを仮定
# =========================================

# 例:
# USD/JPY 150円
# 0.2銭程度を仮定
#
# 0.002円 / 150円
# ≒ 0.0000133

spread_cost = 0.0000133

buy_results[
    "return_after_cost"
] = (
    buy_results[
        "future_return_30m"
    ]
    - spread_cost
)

print("\n========================")
print("スプレッド控除後")
print("========================")

if len(buy_results) > 0:

    print(
        "平均リターン:",
        round(
            buy_results[
                "return_after_cost"
            ].mean() * 100,
            4
        ),
        "%"
    )

    print(
        "利益率プラス:",
        round(
            (
                buy_results[
                    "return_after_cost"
                ] > 0
            ).mean() * 100,
            2
        ),
        "%"
    )


# =========================================
# 29. 1回の取引損益を作る
# =========================================

# BUYシグナルが出た時だけ
# 30分後のリターンを取る

results["strategy_return"] = 0.0

results.loc[
    results["up_probability"] >= 0.60,
    "strategy_return"
] = (
    results.loc[
        results["up_probability"] >= 0.60,
        "future_return_30m"
    ]
    - spread_cost
)


# =========================================
# 30. 累積損益
# =========================================

results["cumulative_return"] = (
    1
    + results["strategy_return"]
).cumprod()


plt.figure(
    figsize=(14, 6)
)

plt.plot(
    results.index,
    results[
        "cumulative_return"
    ]
)

plt.title(
    "BUY Strategy Cumulative Return"
)

plt.xlabel(
    "Time"
)

plt.ylabel(
    "Growth of 1"
)

plt.grid()

plt.show()


# =========================================
# 31. 最大ドローダウン
# =========================================

running_max = (
    results[
        "cumulative_return"
    ]
    .cummax()
)

drawdown = (
    results[
        "cumulative_return"
    ]
    / running_max
    - 1
)

max_drawdown = (
    drawdown.min()
)

print("\n========================")
print("リスク評価")
print("========================")

print(
    "最大ドローダウン:",
    round(
        max_drawdown * 100,
        2
    ),
    "%"
)


# =========================================
# 32. 簡易Sharpe Ratio
# =========================================

strategy_returns = results.loc[
    results["strategy_return"] != 0,
    "strategy_return"
]

if len(strategy_returns) > 1:

    sharpe = (
        strategy_returns.mean()
        / strategy_returns.std()
        * np.sqrt(
            len(strategy_returns)
        )
    )

    print(
        "簡易Sharpe Ratio:",
        round(
            sharpe,
            3
        )
    )


# =========================================
# 33. Walk-Forward検証
# =========================================

print("\n========================")
print("Walk-Forward Test")
print("========================")

# 全データを5分割する
n_splits = 5

split_size = (
    len(data)
    // (n_splits + 1)
)

walk_results = []

for i in range(n_splits):

    # 学習終了位置
    train_end = (
        split_size
        * (i + 1)
    )

    # テスト終了位置
    test_end = (
        train_end
        + split_size
    )

    train_wf = data.iloc[
        :train_end
    ]

    test_wf = data.iloc[
        train_end:test_end
    ]

    # データが足りなければ終了
    if (
        len(train_wf) == 0
        or len(test_wf) == 0
    ):
        continue

    X_train_wf = train_wf[
        features
    ]

    y_train_wf = train_wf[
        "target"
    ]

    X_test_wf = test_wf[
        features
    ]

    y_test_wf = test_wf[
        "target"
    ]

    # 新しいモデル
    wf_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=15,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    # 学習
    wf_model.fit(
        X_train_wf,
        y_train_wf
    )

    # 上昇確率
    wf_prob = (
        wf_model
        .predict_proba(
            X_test_wf
        )[:, 1]
    )

    # BUYシグナル
    buy_mask = (
        wf_prob >= 0.60
    )

    # BUY候補が存在する場合
    if buy_mask.sum() > 0:

        wf_buy_returns = (
            test_wf.loc[
                buy_mask,
                "future_return_30m"
            ]
        )

        wf_win_rate = (
            wf_buy_returns > 0
        ).mean()

        wf_average_return = (
            wf_buy_returns.mean()
        )

        trade_count = (
            buy_mask.sum()
        )

    else:

        wf_win_rate = np.nan
        wf_average_return = np.nan
        trade_count = 0

    # AUC
    try:

        wf_auc = roc_auc_score(
            y_test_wf,
            wf_prob
        )

    except ValueError:

        wf_auc = np.nan

    walk_results.append({
        "period": i + 1,
        "train_size": len(train_wf),
        "test_size": len(test_wf),
        "auc": wf_auc,
        "buy_trades": trade_count,
        "buy_win_rate": wf_win_rate,
        "buy_average_return": wf_average_return
    })


walk_df = pd.DataFrame(
    walk_results
)

walk_df[
    "buy_win_rate"
] *= 100

walk_df[
    "buy_average_return"
] *= 100

print(walk_df)


# =========================================
# 34. Walk-Forward平均
# =========================================

print("\n========================")
print("Walk-Forward平均")
print("========================")

print(
    "平均AUC:",
    round(
        walk_df["auc"].mean(),
        4
    )
)

print(
    "平均BUY勝率:",
    round(
        walk_df[
            "buy_win_rate"
        ].mean(),
        2
    ),
    "%"
)

print(
    "平均BUYリターン:",
    round(
        walk_df[
            "buy_average_return"
        ].mean(),
        4
    ),
    "%"
)

## 元Notebookのセル 8

出典: `FX.ipynb`、0始まりのindex=7。コード内容は変更していません。

In [ ]:
# ============================================================
# 前半
# USD/JPY 5分足
# データ取得 → 特徴量作成 → 3クラス分類 → 学習 → 予測
# ============================================================


# ============================================================
# 1. ライブラリ
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 2. 基本設定
# ============================================================

SYMBOL = "JPY=X"

PERIOD = "60d"
INTERVAL = "5m"

# 5分足6本 = 30分
HOLD_BARS = 6

# ±0.05%以上ならUP/DOWN
MOVE_THRESHOLD = 0.0005

# 売買する最低確率
SIGNAL_THRESHOLD = 0.60

# UPとDOWNの確率差
PROBABILITY_MARGIN = 0.10

# 仮の取引コスト
TRADING_COST = 0.0000133

# 学習とテストの境界に入れる隙間
GAP = HOLD_BARS


# ============================================================
# 3. USD/JPY 5分足を取得
# ============================================================

df = yf.download(
    SYMBOL,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=False,
    progress=False
)

# 列が2段なら1段にする
if df.columns.nlevels > 1:
    df.columns = df.columns.get_level_values(0)

# 日本時間へ変換
if df.index.tz is not None:
    df.index = df.index.tz_convert("Asia/Tokyo")

print("取得した5分足:", len(df))


# ============================================================
# 4. リターン
# ============================================================

df["return_5m"] = df["Close"].pct_change(1)
df["return_15m"] = df["Close"].pct_change(3)
df["return_30m"] = df["Close"].pct_change(6)
df["return_1h"] = df["Close"].pct_change(12)
df["return_2h"] = df["Close"].pct_change(24)


# ============================================================
# 5. 移動平均
# ============================================================

df["MA5"] = df["Close"].rolling(5).mean()
df["MA20"] = df["Close"].rolling(20).mean()
df["MA50"] = df["Close"].rolling(50).mean()

df["MA5_distance"] = df["Close"] / df["MA5"] - 1
df["MA20_distance"] = df["Close"] / df["MA20"] - 1
df["MA50_distance"] = df["Close"] / df["MA50"] - 1

df["MA5_slope"] = df["MA5"].pct_change(3)
df["MA20_slope"] = df["MA20"].pct_change(3)
df["MA50_slope"] = df["MA50"].pct_change(3)


# ============================================================
# 6. ローソク足特徴
# ============================================================

df["body"] = (
    abs(df["Close"] - df["Open"])
    / df["Open"]
)

df["range"] = (
    (df["High"] - df["Low"])
    / df["Close"]
)

df["upper_wick"] = (
    df["High"]
    - df[["Open", "Close"]].max(axis=1)
) / df["Close"]

df["lower_wick"] = (
    df[["Open", "Close"]].min(axis=1)
    - df["Low"]
) / df["Close"]

df["bullish"] = (
    df["Close"] > df["Open"]
).astype(int)


# ============================================================
# 7. ボラティリティ
# ============================================================

df["volatility_1h"] = (
    df["return_5m"]
    .rolling(12)
    .std()
)

df["volatility_2h"] = (
    df["return_5m"]
    .rolling(24)
    .std()
)

df["volatility_4h"] = (
    df["return_5m"]
    .rolling(48)
    .std()
)


# ============================================================
# 8. RSI
# ============================================================

delta = df["Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df["RSI"] = (
    100
    - 100 / (1 + rs)
)


# ============================================================
# 9. 高値・安値からの距離
# ============================================================

df["high_1h"] = (
    df["High"]
    .rolling(12)
    .max()
)

df["low_1h"] = (
    df["Low"]
    .rolling(12)
    .min()
)

df["distance_high_1h"] = (
    df["Close"]
    / df["high_1h"]
    - 1
)

df["distance_low_1h"] = (
    df["Close"]
    / df["low_1h"]
    - 1
)


# ============================================================
# 10. 時間情報
# ============================================================

df["hour"] = df.index.hour
df["weekday"] = df.index.dayofweek

# 23時と0時を近い値として表現する
df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)


# ============================================================
# 11. 現実的な将来リターン
# ============================================================

# 現在足が確定した後、
# 次の5分足のOpenでエントリー
df["entry_price"] = (
    df["Open"].shift(-1)
)

# 30分後に決済
df["exit_price"] = (
    df["Close"].shift(-HOLD_BARS)
)

df["future_return"] = (
    df["exit_price"]
    / df["entry_price"]
    - 1
)


# ============================================================
# 12. 3クラス正解
# ============================================================

# 0 = DOWN
# 1 = WAIT
# 2 = UP

df["target"] = 1

df.loc[
    df["future_return"] > MOVE_THRESHOLD,
    "target"
] = 2

df.loc[
    df["future_return"] < -MOVE_THRESHOLD,
    "target"
] = 0


# ============================================================
# 13. AIに渡す特徴量
# ============================================================

features = [

    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",
    "return_2h",

    "MA5_distance",
    "MA20_distance",
    "MA50_distance",

    "MA5_slope",
    "MA20_slope",
    "MA50_slope",

    "body",
    "range",
    "upper_wick",
    "lower_wick",
    "bullish",

    "volatility_1h",
    "volatility_2h",
    "volatility_4h",

    "RSI",

    "distance_high_1h",
    "distance_low_1h",

    "weekday",

    "hour_sin",
    "hour_cos"
]


# ============================================================
# 14. データ整形
# ============================================================

required_columns = (
    features
    + [
        "target",
        "future_return",
        "entry_price",
        "exit_price"
    ]
)

data = (
    df[required_columns]
    .dropna()
    .copy()
)

data["target"] = (
    data["target"]
    .astype(int)
)

print("機械学習に使用可能:", len(data))

print("\nクラス割合")
print(
    data["target"]
    .value_counts(normalize=True)
    .sort_index()
)


# ============================================================
# 15. 学習80% / テスト20%
# ============================================================

split = int(
    len(data) * 0.80
)

train = (
    data.iloc[
        :split - GAP
    ]
)

test = (
    data.iloc[
        split:
    ]
)

X_train = train[features]
y_train = train["target"]

X_test = test[features]
y_test = test["target"]

print("\n学習:", len(train))
print("Gap:", GAP)
print("テスト:", len(test))


# ============================================================
# 16. Random Forest
# ============================================================

model = RandomForestClassifier(

    n_estimators=500,

    max_depth=8,

    min_samples_leaf=20,

    max_features="sqrt",

    class_weight="balanced",

    random_state=42,

    n_jobs=-1
)


# ============================================================
# 17. 学習
# ============================================================

model.fit(
    X_train,
    y_train
)


# ============================================================
# 18. 予測
# ============================================================

pred = model.predict(
    X_test
)

prob = model.predict_proba(
    X_test
)


# ============================================================
# 19. クラス確率を取り出す
# ============================================================

print(
    "\nモデルクラス:",
    model.classes_
)

class_to_col = {
    class_label: i
    for i, class_label
    in enumerate(model.classes_)
}

p_down = prob[
    :,
    class_to_col[0]
]

p_wait = prob[
    :,
    class_to_col[1]
]

p_up = prob[
    :,
    class_to_col[2]
]


# ============================================================
# 20. 分類性能
# ============================================================

accuracy = accuracy_score(
    y_test,
    pred
)

print("\n==============================")
print("3クラス分類性能")
print("==============================")

print(
    "Accuracy:",
    round(accuracy, 4)
)

print(
    classification_report(
        y_test,
        pred,
        target_names=[
            "DOWN",
            "WAIT",
            "UP"
        ]
    )
)

print("Confusion Matrix")

print(
    confusion_matrix(
        y_test,
        pred
    )
)


# ============================================================
# 21. テスト結果を保存
# ============================================================

results = test.copy()

results["p_down"] = p_down
results["p_wait"] = p_wait
results["p_up"] = p_up

print("\n前半完了")
print("次に後半コードを実行してください")

# ============================================================
# 後半
# シグナル生成 → 現実的バックテスト → Walk-Forward
# ============================================================


# ============================================================
# 22. BUY / SELLシグナル
# ============================================================

results["signal"] = 0


# BUY条件
buy_condition = (

    (results["p_up"] >= SIGNAL_THRESHOLD)

    &

    (
        results["p_up"]
        - results["p_down"]
        >= PROBABILITY_MARGIN
    )
)


# SELL条件
sell_condition = (

    (results["p_down"] >= SIGNAL_THRESHOLD)

    &

    (
        results["p_down"]
        - results["p_up"]
        >= PROBABILITY_MARGIN
    )
)


results.loc[
    buy_condition,
    "signal"
] = 1


results.loc[
    sell_condition,
    "signal"
] = -1


print("\n==============================")
print("シグナル")
print("==============================")

print(
    "BUY候補:",
    (results["signal"] == 1).sum()
)

print(
    "SELL候補:",
    (results["signal"] == -1).sum()
)


# ============================================================
# 23. 現実的バックテスト
# ============================================================

# 1ポジションだけ保有
# 30分保有中は新しいシグナルを無視

trades = []

i = 0

while i < len(results):

    row = results.iloc[i]

    signal = row["signal"]


    # WAIT
    if signal == 0:

        i += 1
        continue


    # データ不足なら終了
    if i + HOLD_BARS >= len(results):

        break


    entry_price = row["entry_price"]
    exit_price = row["exit_price"]


    # BUY
    if signal == 1:

        gross_return = (
            exit_price
            / entry_price
            - 1
        )

        direction = "BUY"


    # SELL
    else:

        gross_return = (
            entry_price
            / exit_price
            - 1
        )

        direction = "SELL"


    # コストを引く
    net_return = (
        gross_return
        - TRADING_COST
    )


    trades.append({

        "signal_time":
            results.index[i],

        "direction":
            direction,

        "entry_price":
            entry_price,

        "exit_price":
            exit_price,

        "p_up":
            row["p_up"],

        "p_wait":
            row["p_wait"],

        "p_down":
            row["p_down"],

        "gross_return":
            gross_return,

        "net_return":
            net_return
    })


    # 30分分飛ばす
    i += HOLD_BARS


# ============================================================
# 24. 取引結果
# ============================================================

trades_df = pd.DataFrame(
    trades
)

print("\n==============================")
print("現実的バックテスト")
print("==============================")


if len(trades_df) == 0:

    print(
        "取引シグナルなし"
    )

else:

    print(
        "総取引数:",
        len(trades_df)
    )

    print(
        "勝率:",
        round(
            (
                trades_df["net_return"] > 0
            ).mean()
            * 100,
            2
        ),
        "%"
    )

    print(
        "平均リターン:",
        round(
            trades_df[
                "net_return"
            ].mean()
            * 100,
            4
        ),
        "%"
    )

    print(
        "中央値:",
        round(
            trades_df[
                "net_return"
            ].median()
            * 100,
            4
        ),
        "%"
    )

    print(
        "最大利益:",
        round(
            trades_df[
                "net_return"
            ].max()
            * 100,
            4
        ),
        "%"
    )

    print(
        "最大損失:",
        round(
            trades_df[
                "net_return"
            ].min()
            * 100,
            4
        ),
        "%"
    )


# ============================================================
# 25. BUY / SELL別
# ============================================================

if len(trades_df) > 0:

    for direction in [
        "BUY",
        "SELL"
    ]:

        subset = trades_df[
            trades_df["direction"]
            == direction
        ]

        if len(subset) == 0:
            continue

        print(
            "\n=============================="
        )

        print(
            direction,
            "分析"
        )

        print(
            "件数:",
            len(subset)
        )

        print(
            "勝率:",
            round(
                (
                    subset["net_return"] > 0
                ).mean()
                * 100,
                2
            ),
            "%"
        )

        print(
            "平均リターン:",
            round(
                subset[
                    "net_return"
                ].mean()
                * 100,
                4
            ),
            "%"
        )


# ============================================================
# 26. 累積損益
# ============================================================

if len(trades_df) > 0:

    trades_df[
        "equity"
    ] = (
        1
        + trades_df[
            "net_return"
        ]
    ).cumprod()


    plt.figure(
        figsize=(14, 6)
    )

    plt.plot(
        trades_df[
            "signal_time"
        ],
        trades_df[
            "equity"
        ]
    )

    plt.title(
        "Realistic Strategy Equity Curve"
    )

    plt.xlabel("Time")
    plt.ylabel("Growth of 1")

    plt.grid()

    plt.show()


# ============================================================
# 27. 最大ドローダウン
# ============================================================

if len(trades_df) > 0:

    running_max = (
        trades_df[
            "equity"
        ]
        .cummax()
    )

    drawdown = (
        trades_df[
            "equity"
        ]
        / running_max
        - 1
    )

    max_drawdown = (
        drawdown.min()
    )

    print(
        "\n最大ドローダウン:",
        round(
            max_drawdown
            * 100,
            2
        ),
        "%"
    )


# ============================================================
# 28. Profit Factor
# ============================================================

if len(trades_df) > 0:

    positive_sum = (
        trades_df.loc[
            trades_df[
                "net_return"
            ] > 0,
            "net_return"
        ]
        .sum()
    )

    negative_sum = abs(
        trades_df.loc[
            trades_df[
                "net_return"
            ] < 0,
            "net_return"
        ]
        .sum()
    )

    if negative_sum > 0:

        profit_factor = (
            positive_sum
            / negative_sum
        )

        print(
            "Profit Factor:",
            round(
                profit_factor,
                3
            )
        )


# ============================================================
# 29. 特徴量重要度
# ============================================================

importance = pd.Series(

    model.feature_importances_,

    index=features

).sort_values(
    ascending=False
)

print("\n==============================")
print("特徴量重要度")
print("==============================")

print(importance)

importance.plot(
    kind="bar",
    figsize=(13, 6)
)

plt.title(
    "Feature Importance"
)

plt.ylabel(
    "Importance"
)

plt.grid()

plt.show()


# ============================================================
# 30. Walk-Forward
# ============================================================

print("\n==============================")
print("Walk-Forward")
print("==============================")

N_SPLITS = 5

block_size = (
    len(data)
    // (N_SPLITS + 1)
)

walk_results = []


for fold in range(N_SPLITS):

    train_end = (
        block_size
        * (fold + 1)
    )

    test_start = (
        train_end
        + GAP
    )

    test_end = (
        test_start
        + block_size
    )

    if test_end > len(data):
        test_end = len(data)


    train_wf = (
        data.iloc[
            :train_end
        ]
    )

    test_wf = (
        data.iloc[
            test_start:test_end
        ]
    )


    if (
        len(train_wf) == 0
        or
        len(test_wf) == 0
    ):
        continue


    X_train_wf = (
        train_wf[
            features
        ]
    )

    y_train_wf = (
        train_wf[
            "target"
        ]
    )

    X_test_wf = (
        test_wf[
            features
        ]
    )


    wf_model = RandomForestClassifier(

        n_estimators=400,

        max_depth=8,

        min_samples_leaf=20,

        max_features="sqrt",

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    )


    wf_model.fit(
        X_train_wf,
        y_train_wf
    )


    wf_prob = (
        wf_model
        .predict_proba(
            X_test_wf
        )
    )


    wf_classes = {
        c: j
        for j, c
        in enumerate(
            wf_model.classes_
        )
    }


    wf_p_down = (
        wf_prob[
            :,
            wf_classes[0]
        ]
    )

    wf_p_up = (
        wf_prob[
            :,
            wf_classes[2]
        ]
    )


    wf_signals = np.zeros(
        len(test_wf)
    )


    wf_buy = (

        (wf_p_up >= SIGNAL_THRESHOLD)

        &

        (
            wf_p_up
            - wf_p_down
            >= PROBABILITY_MARGIN
        )
    )


    wf_sell = (

        (wf_p_down >= SIGNAL_THRESHOLD)

        &

        (
            wf_p_down
            - wf_p_up
            >= PROBABILITY_MARGIN
        )
    )


    wf_signals[
        wf_buy
    ] = 1

    wf_signals[
        wf_sell
    ] = -1


    # ----------------------------------------
    # 非重複バックテスト
    # ----------------------------------------

    wf_returns = []

    i = 0

    while i < len(test_wf):

        signal = (
            wf_signals[i]
        )

        if signal == 0:

            i += 1
            continue


        row = (
            test_wf.iloc[i]
        )


        if signal == 1:

            r = (
                row["future_return"]
                - TRADING_COST
            )

        else:

            r = (
                -row["future_return"]
                - TRADING_COST
            )


        wf_returns.append(
            r
        )


        i += HOLD_BARS


    wf_returns = np.array(
        wf_returns
    )


    if len(wf_returns) > 0:

        wf_win_rate = (
            wf_returns > 0
        ).mean()

        wf_average = (
            wf_returns.mean()
        )

    else:

        wf_win_rate = np.nan
        wf_average = np.nan


    walk_results.append({

        "fold":
            fold + 1,

        "train_size":
            len(train_wf),

        "test_size":
            len(test_wf),

        "trades":
            len(wf_returns),

        "win_rate":
            wf_win_rate,

        "average_return":
            wf_average
    })


walk_df = pd.DataFrame(
    walk_results
)

walk_df[
    "win_rate"
] *= 100

walk_df[
    "average_return"
] *= 100


print(
    walk_df
)


print("\n==============================")
print("Walk-Forward平均")
print("==============================")


print(
    "平均勝率:",
    round(
        walk_df[
            "win_rate"
        ].mean(),
        2
    ),
    "%"
)


print(
    "平均リターン:",
    round(
        walk_df[
            "average_return"
        ].mean(),
        4
    ),
    "%"
)


print(
    "合計取引数:",
    walk_df[
        "trades"
    ].sum()
)


# ============================================================
# 31. 最新予測
# ============================================================

latest_X = (
    data[
        features
    ]
    .iloc[[-1]]
)

latest_prob = (
    model.predict_proba(
        latest_X
    )[0]
)

latest_map = {

    c: latest_prob[i]

    for i, c
    in enumerate(
        model.classes_
    )
}


print("\n==============================")
print("最新予測")
print("==============================")


print(
    "DOWN:",
    round(
        latest_map.get(0, 0)
        * 100,
        2
    ),
    "%"
)


print(
    "WAIT:",
    round(
        latest_map.get(1, 0)
        * 100,
        2
    ),
    "%"
)


print(
    "UP:",
    round(
        latest_map.get(2, 0)
        * 100,
        2
    ),
    "%"
)

## 元Notebookのセル 9

出典: `FX.ipynb`、0始まりのindex=8。コード内容は変更していません。

In [ ]:
# ============================================================
# USD/JPY 5分足
# 2段階モデル
# AI① MOVE / WAIT
# AI② UP / DOWN
# 現実的バックテスト + Walk-Forward
# ============================================================


# ============================================================
# 1. ライブラリ
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report
)


# ============================================================
# 2. 基本設定
# ============================================================

SYMBOL = "JPY=X"

PERIOD = "60d"
INTERVAL = "5m"

# 30分 = 5分足6本
HOLD_BARS = 6

# ±0.05%以上動けばMOVE
MOVE_THRESHOLD = 0.0005

# AI①
# これ以上なら「動きそう」と判断
MOVE_PROB_THRESHOLD = 0.65

# AI②
# UP/DOWNどちらかがこれ以上なら売買
DIRECTION_PROB_THRESHOLD = 0.60

# UPとDOWNの確率差
DIRECTION_MARGIN = 0.10

# 仮の取引コスト
TRADING_COST = 0.0000133

# 学習とテストの境界Gap
GAP = HOLD_BARS


# ============================================================
# 3. USD/JPY 5分足を取得
# ============================================================

df = yf.download(
    SYMBOL,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=False,
    progress=False
)

# yfinanceの列が2段なら1段へ
if df.columns.nlevels > 1:
    df.columns = df.columns.get_level_values(0)

# 日本時間にする
if df.index.tz is not None:
    df.index = df.index.tz_convert("Asia/Tokyo")

print("取得した5分足:", len(df))


# ============================================================
# 4. リターン特徴量
# ============================================================

df["return_5m"] = df["Close"].pct_change(1)

df["return_15m"] = df["Close"].pct_change(3)

df["return_30m"] = df["Close"].pct_change(6)

df["return_1h"] = df["Close"].pct_change(12)

df["return_2h"] = df["Close"].pct_change(24)


# ============================================================
# 5. 移動平均
# ============================================================

df["MA5"] = (
    df["Close"]
    .rolling(5)
    .mean()
)

df["MA20"] = (
    df["Close"]
    .rolling(20)
    .mean()
)

df["MA50"] = (
    df["Close"]
    .rolling(50)
    .mean()
)


# 現在値とMAの距離
df["MA5_distance"] = (
    df["Close"]
    / df["MA5"]
    - 1
)

df["MA20_distance"] = (
    df["Close"]
    / df["MA20"]
    - 1
)

df["MA50_distance"] = (
    df["Close"]
    / df["MA50"]
    - 1
)


# MA傾き
df["MA5_slope"] = (
    df["MA5"]
    .pct_change(3)
)

df["MA20_slope"] = (
    df["MA20"]
    .pct_change(3)
)

df["MA50_slope"] = (
    df["MA50"]
    .pct_change(3)
)


# ============================================================
# 6. ローソク足特徴
# ============================================================

# 実体
df["body"] = (
    abs(
        df["Close"]
        - df["Open"]
    )
    / df["Open"]
)


# 高値-安値
df["range"] = (
    df["High"]
    - df["Low"]
) / df["Close"]


# 上ヒゲ
df["upper_wick"] = (
    df["High"]
    - df[
        ["Open", "Close"]
    ].max(axis=1)
) / df["Close"]


# 下ヒゲ
df["lower_wick"] = (
    df[
        ["Open", "Close"]
    ].min(axis=1)
    - df["Low"]
) / df["Close"]


# 陽線か
df["bullish"] = (
    df["Close"]
    > df["Open"]
).astype(int)


# ============================================================
# 7. ボラティリティ
# ============================================================

df["volatility_1h"] = (
    df["return_5m"]
    .rolling(12)
    .std()
)

df["volatility_2h"] = (
    df["return_5m"]
    .rolling(24)
    .std()
)

df["volatility_4h"] = (
    df["return_5m"]
    .rolling(48)
    .std()
)


# ============================================================
# 8. RSI
# ============================================================

delta = (
    df["Close"]
    .diff()
)

gain = (
    delta
    .clip(lower=0)
)

loss = (
    -delta
    .clip(upper=0)
)

avg_gain = (
    gain
    .rolling(14)
    .mean()
)

avg_loss = (
    loss
    .rolling(14)
    .mean()
)

rs = (
    avg_gain
    / avg_loss
)

df["RSI"] = (
    100
    - 100 / (1 + rs)
)


# ============================================================
# 9. 高値安値からの距離
# ============================================================

df["high_1h"] = (
    df["High"]
    .rolling(12)
    .max()
)

df["low_1h"] = (
    df["Low"]
    .rolling(12)
    .min()
)


df["distance_high_1h"] = (
    df["Close"]
    / df["high_1h"]
    - 1
)

df["distance_low_1h"] = (
    df["Close"]
    / df["low_1h"]
    - 1
)


# ============================================================
# 10. 時間特徴
# ============================================================

df["hour"] = df.index.hour

df["weekday"] = (
    df.index.dayofweek
)

df["hour_sin"] = np.sin(
    2
    * np.pi
    * df["hour"]
    / 24
)

df["hour_cos"] = np.cos(
    2
    * np.pi
    * df["hour"]
    / 24
)


# ============================================================
# 11. 現実的な将来リターン
# ============================================================

# 現在足確定後
# 次の5分足Openでエントリー
df["entry_price"] = (
    df["Open"]
    .shift(-1)
)

# 30分後に決済
df["exit_price"] = (
    df["Close"]
    .shift(-HOLD_BARS)
)

df["future_return"] = (
    df["exit_price"]
    / df["entry_price"]
    - 1
)


# ============================================================
# 12. AI① MOVE用ターゲット
# ============================================================

# 30分後に±0.05%以上なら1
# それ以外なら0

df["move_target"] = (
    abs(
        df["future_return"]
    )
    > MOVE_THRESHOLD
).astype(int)


# ============================================================
# 13. AI② DIRECTION用ターゲット
# ============================================================

# UP = 1
# DOWN = 0
#
# ここはMOVEしたケースだけで学習する

df["direction_target"] = np.where(
    df["future_return"] > 0,
    1,
    0
)


# ============================================================
# 14. AI① MOVEモデル用特徴量
# ============================================================

move_features = [

    "volatility_1h",
    "volatility_2h",
    "volatility_4h",

    "range",
    "body",

    "return_5m",
    "return_15m",
    "return_30m",

    "MA20_slope",
    "MA50_slope",

    "distance_high_1h",
    "distance_low_1h",

    "hour_sin",
    "hour_cos",

    "weekday"
]


# ============================================================
# 15. AI② 方向モデル用特徴量
# ============================================================

direction_features = [

    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",
    "return_2h",

    "MA5_distance",
    "MA20_distance",
    "MA50_distance",

    "MA5_slope",
    "MA20_slope",
    "MA50_slope",

    "RSI",

    "bullish",
    "body",
    "upper_wick",
    "lower_wick",

    "distance_high_1h",
    "distance_low_1h",

    "volatility_1h",

    "hour_sin",
    "hour_cos",

    "weekday"
]


# ============================================================
# 16. データ整形
# ============================================================

all_features = list(
    set(
        move_features
        + direction_features
    )
)

required_columns = (
    all_features
    + [
        "move_target",
        "direction_target",
        "future_return",
        "entry_price",
        "exit_price"
    ]
)

data = (
    df[
        required_columns
    ]
    .dropna()
    .copy()
)

print(
    "機械学習使用可能:",
    len(data)
)


print("\nMOVE割合")

print(
    data[
        "move_target"
    ]
    .value_counts(
        normalize=True
    )
)


# ============================================================
# 17. 学習80% / テスト20%
# ============================================================

split = int(
    len(data)
    * 0.80
)

train = (
    data.iloc[
        :split - GAP
    ]
)

test = (
    data.iloc[
        split:
    ]
)


# ============================================================
# 18. AI① MOVEモデル
# ============================================================

X_move_train = (
    train[
        move_features
    ]
)

y_move_train = (
    train[
        "move_target"
    ]
)

X_move_test = (
    test[
        move_features
    ]
)

y_move_test = (
    test[
        "move_target"
    ]
)


move_model = RandomForestClassifier(

    n_estimators=500,

    max_depth=8,

    min_samples_leaf=20,

    max_features="sqrt",

    class_weight="balanced",

    random_state=42,

    n_jobs=-1
)


move_model.fit(
    X_move_train,
    y_move_train
)


move_prob = (
    move_model
    .predict_proba(
        X_move_test
    )[:, 1]
)


move_pred = (
    move_model
    .predict(
        X_move_test
    )
)


print("\n==============================")
print("AI① MOVEモデル")
print("==============================")

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_move_test,
            move_pred
        ),
        4
    )
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(
            y_move_test,
            move_prob
        ),
        4
    )
)

print(
    classification_report(
        y_move_test,
        move_pred
    )
)


# ============================================================
# 19. AI② 方向モデル
# ============================================================

# MOVEした学習データだけ使う
direction_train = (
    train[
        train[
            "move_target"
        ] == 1
    ]
)


X_direction_train = (
    direction_train[
        direction_features
    ]
)

y_direction_train = (
    direction_train[
        "direction_target"
    ]
)


direction_model = RandomForestClassifier(

    n_estimators=500,

    max_depth=8,

    min_samples_leaf=15,

    max_features="sqrt",

    class_weight="balanced",

    random_state=42,

    n_jobs=-1
)


direction_model.fit(
    X_direction_train,
    y_direction_train
)


# テストは全ケースに確率を出す
direction_prob = (
    direction_model
    .predict_proba(
        test[
            direction_features
        ]
    )
)


direction_classes = {
    c: i
    for i, c
    in enumerate(
        direction_model.classes_
    )
}


p_down = (
    direction_prob[
        :,
        direction_classes[0]
    ]
)

p_up = (
    direction_prob[
        :,
        direction_classes[1]
    ]
)


# ============================================================
# 20. テスト結果DataFrame
# ============================================================

results = test.copy()

results["p_move"] = (
    move_prob
)

results["p_up"] = (
    p_up
)

results["p_down"] = (
    p_down
)


# ============================================================
# 21. シグナル
# ============================================================

results["signal"] = 0


# BUY条件
buy_condition = (

    (
        results["p_move"]
        >= MOVE_PROB_THRESHOLD
    )

    &

    (
        results["p_up"]
        >= DIRECTION_PROB_THRESHOLD
    )

    &

    (
        results["p_up"]
        - results["p_down"]
        >= DIRECTION_MARGIN
    )
)


# SELL条件
sell_condition = (

    (
        results["p_move"]
        >= MOVE_PROB_THRESHOLD
    )

    &

    (
        results["p_down"]
        >= DIRECTION_PROB_THRESHOLD
    )

    &

    (
        results["p_down"]
        - results["p_up"]
        >= DIRECTION_MARGIN
    )
)


results.loc[
    buy_condition,
    "signal"
] = 1

results.loc[
    sell_condition,
    "signal"
] = -1


print("\n==============================")
print("二段階シグナル")
print("==============================")

print(
    "BUY候補:",
    (
        results[
            "signal"
        ] == 1
    ).sum()
)

print(
    "SELL候補:",
    (
        results[
            "signal"
        ] == -1
    ).sum()
)


# ============================================================
# 22. 現実的バックテスト
# ============================================================

trades = []

i = 0

while i < len(results):

    row = (
        results.iloc[i]
    )

    signal = (
        row["signal"]
    )


    if signal == 0:

        i += 1
        continue


    if (
        i + HOLD_BARS
        >= len(results)
    ):

        break


    entry_price = (
        row["entry_price"]
    )

    exit_price = (
        row["exit_price"]
    )


    if signal == 1:

        gross_return = (
            exit_price
            / entry_price
            - 1
        )

        direction = "BUY"


    else:

        gross_return = (
            entry_price
            / exit_price
            - 1
        )

        direction = "SELL"


    net_return = (
        gross_return
        - TRADING_COST
    )


    trades.append({

        "signal_time":
            results.index[i],

        "direction":
            direction,

        "p_move":
            row["p_move"],

        "p_up":
            row["p_up"],

        "p_down":
            row["p_down"],

        "entry_price":
            entry_price,

        "exit_price":
            exit_price,

        "net_return":
            net_return
    })


    # 保有中は新規取引しない
    i += HOLD_BARS


trades_df = pd.DataFrame(
    trades
)


# ============================================================
# 23. バックテスト結果
# ============================================================

print("\n==============================")
print("バックテスト")
print("==============================")


if len(trades_df) == 0:

    print(
        "取引なし"
    )

else:

    print(
        "総取引数:",
        len(trades_df)
    )

    print(
        "勝率:",
        round(
            (
                trades_df[
                    "net_return"
                ] > 0
            ).mean()
            * 100,
            2
        ),
        "%"
    )

    print(
        "平均リターン:",
        round(
            trades_df[
                "net_return"
            ].mean()
            * 100,
            4
        ),
        "%"
    )

    print(
        "中央値:",
        round(
            trades_df[
                "net_return"
            ].median()
            * 100,
            4
        ),
        "%"
    )


# ============================================================
# 24. BUY / SELL別
# ============================================================

if len(trades_df) > 0:

    for direction in [
        "BUY",
        "SELL"
    ]:

        subset = (
            trades_df[
                trades_df[
                    "direction"
                ] == direction
            ]
        )

        if len(subset) == 0:
            continue


        print(
            "\n",
            direction
        )

        print(
            "件数:",
            len(subset)
        )

        print(
            "勝率:",
            round(
                (
                    subset[
                        "net_return"
                    ] > 0
                ).mean()
                * 100,
                2
            ),
            "%"
        )

        print(
            "平均:",
            round(
                subset[
                    "net_return"
                ].mean()
                * 100,
                4
            ),
            "%"
        )


# ============================================================
# 25. 累積損益
# ============================================================

if len(trades_df) > 0:

    trades_df["equity"] = (

        1
        + trades_df[
            "net_return"
        ]

    ).cumprod()


    plt.figure(
        figsize=(14, 6)
    )

    plt.plot(
        trades_df[
            "signal_time"
        ],
        trades_df[
            "equity"
        ]
    )

    plt.title(
        "Two Stage Model Equity"
    )

    plt.xlabel(
        "Time"
    )

    plt.ylabel(
        "Growth of 1"
    )

    plt.grid()

    plt.show()


# ============================================================
# 26. 最大ドローダウン
# ============================================================

if len(trades_df) > 0:

    running_max = (
        trades_df[
            "equity"
        ]
        .cummax()
    )

    drawdown = (

        trades_df[
            "equity"
        ]

        / running_max

        - 1
    )

    max_drawdown = (
        drawdown.min()
    )

    print(
        "\n最大ドローダウン:",
        round(
            max_drawdown
            * 100,
            2
        ),
        "%"
    )


# ============================================================
# 27. Profit Factor
# ============================================================

if len(trades_df) > 0:

    total_profit = (
        trades_df.loc[
            trades_df[
                "net_return"
            ] > 0,
            "net_return"
        ]
        .sum()
    )

    total_loss = abs(

        trades_df.loc[
            trades_df[
                "net_return"
            ] < 0,
            "net_return"
        ]
        .sum()

    )


    if total_loss > 0:

        profit_factor = (
            total_profit
            / total_loss
        )

        print(
            "Profit Factor:",
            round(
                profit_factor,
                3
            )
        )


# ============================================================
# 28. AI① 特徴量重要度
# ============================================================

move_importance = pd.Series(

    move_model.feature_importances_,

    index=move_features

).sort_values(
    ascending=False
)

print("\n==============================")
print("MOVE特徴量重要度")
print("==============================")

print(
    move_importance
)


# ============================================================
# 29. AI② 特徴量重要度
# ============================================================

direction_importance = pd.Series(

    direction_model.feature_importances_,

    index=direction_features

).sort_values(
    ascending=False
)

print("\n==============================")
print("方向特徴量重要度")
print("==============================")

print(
    direction_importance
)


# ============================================================
# 30. Walk-Forward
# ============================================================

print("\n==============================")
print("Walk-Forward")
print("==============================")


N_SPLITS = 5

block_size = (
    len(data)
    // (N_SPLITS + 1)
)

walk_results = []


for fold in range(N_SPLITS):


    train_end = (
        block_size
        * (fold + 1)
    )


    test_start = (
        train_end
        + GAP
    )


    test_end = (
        test_start
        + block_size
    )


    if test_end > len(data):

        test_end = len(data)


    train_wf = (
        data.iloc[
            :train_end
        ]
    )


    test_wf = (
        data.iloc[
            test_start:test_end
        ]
    )


    if (
        len(train_wf) == 0
        or
        len(test_wf) == 0
    ):

        continue


    # ----------------------------------------
    # AI①
    # ----------------------------------------

    wf_move_model = RandomForestClassifier(

        n_estimators=300,

        max_depth=8,

        min_samples_leaf=20,

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    )


    wf_move_model.fit(

        train_wf[
            move_features
        ],

        train_wf[
            "move_target"
        ]
    )


    wf_p_move = (
        wf_move_model
        .predict_proba(
            test_wf[
                move_features
            ]
        )[:, 1]
    )


    # ----------------------------------------
    # AI②
    # ----------------------------------------

    wf_direction_train = (

        train_wf[
            train_wf[
                "move_target"
            ] == 1
        ]

    )


    wf_direction_model = RandomForestClassifier(

        n_estimators=300,

        max_depth=8,

        min_samples_leaf=15,

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    )


    wf_direction_model.fit(

        wf_direction_train[
            direction_features
        ],

        wf_direction_train[
            "direction_target"
        ]
    )


    wf_direction_prob = (
        wf_direction_model
        .predict_proba(
            test_wf[
                direction_features
            ]
        )
    )


    wf_classes = {

        c: j

        for j, c

        in enumerate(
            wf_direction_model.classes_
        )
    }


    wf_p_down = (
        wf_direction_prob[
            :,
            wf_classes[0]
        ]
    )


    wf_p_up = (
        wf_direction_prob[
            :,
            wf_classes[1]
        ]
    )


    # ----------------------------------------
    # シグナル
    # ----------------------------------------

    wf_signals = np.zeros(
        len(test_wf)
    )


    wf_buy = (

        (wf_p_move >= MOVE_PROB_THRESHOLD)

        &

        (wf_p_up >= DIRECTION_PROB_THRESHOLD)

        &

        (
            wf_p_up
            - wf_p_down
            >= DIRECTION_MARGIN
        )
    )


    wf_sell = (

        (wf_p_move >= MOVE_PROB_THRESHOLD)

        &

        (wf_p_down >= DIRECTION_PROB_THRESHOLD)

        &

        (
            wf_p_down
            - wf_p_up
            >= DIRECTION_MARGIN
        )
    )


    wf_signals[
        wf_buy
    ] = 1


    wf_signals[
        wf_sell
    ] = -1


    # ----------------------------------------
    # 非重複取引
    # ----------------------------------------

    wf_returns = []

    i = 0


    while i < len(test_wf):


        signal = (
            wf_signals[i]
        )


        if signal == 0:

            i += 1
            continue


        row = (
            test_wf.iloc[i]
        )


        if signal == 1:

            r = (
                row[
                    "future_return"
                ]
                - TRADING_COST
            )


        else:

            r = (
                -row[
                    "future_return"
                ]
                - TRADING_COST
            )


        wf_returns.append(
            r
        )


        i += HOLD_BARS


    wf_returns = np.array(
        wf_returns
    )


    if len(wf_returns) > 0:

        win_rate = (
            wf_returns
            > 0
        ).mean()

        average_return = (
            wf_returns.mean()
        )


    else:

        win_rate = np.nan
        average_return = np.nan


    walk_results.append({

        "fold":
            fold + 1,

        "train_size":
            len(train_wf),

        "test_size":
            len(test_wf),

        "trades":
            len(wf_returns),

        "win_rate":
            win_rate,

        "average_return":
            average_return
    })


walk_df = pd.DataFrame(
    walk_results
)


walk_df[
    "win_rate"
] *= 100

walk_df[
    "average_return"
] *= 100


print(
    walk_df
)


print("\n==============================")
print("Walk-Forward平均")
print("==============================")


print(
    "平均勝率:",
    round(
        walk_df[
            "win_rate"
        ].mean(),
        2
    ),
    "%"
)


print(
    "平均リターン:",
    round(
        walk_df[
            "average_return"
        ].mean(),
        4
    ),
    "%"
)


print(
    "合計取引数:",
    walk_df[
        "trades"
    ].sum()
)


# ============================================================
# 31. 最新予測
# ============================================================

latest_data = (
    data.iloc[[-1]]
)


latest_move_prob = (

    move_model
    .predict_proba(
        latest_data[
            move_features
        ]
    )[0, 1]

)


latest_direction_prob = (

    direction_model
    .predict_proba(
        latest_data[
            direction_features
        ]
    )[0]

)


latest_direction_map = {

    c:
        latest_direction_prob[i]

    for i, c

    in enumerate(
        direction_model.classes_
    )

}


print("\n==============================")
print("最新予測")
print("==============================")


print(
    "MOVE:",
    round(
        latest_move_prob
        * 100,
        2
    ),
    "%"
)


print(
    "DOWN:",
    round(
        latest_direction_map.get(
            0,
            0
        )
        * 100,
        2
    ),
    "%"
)


print(
    "UP:",
    round(
        latest_direction_map.get(
            1,
            0
        )
        * 100,
        2
    ),
    "%"
)